# 🚀 ARTHA V1 — Google Colab Runner

This notebook clones the ARTHA V1 repo, installs dependencies, and runs the full algo trading system.

**Run cells top to bottom. No manual steps needed.**

---

## 1️⃣ Clone the Repository

⚠️ **IMPORTANT**: Replace `YOUR_GITHUB_USERNAME` with your actual GitHub username below.

In [ ]:
# =====================================================
# STEP 1: Clone repo from GitHub
# =====================================================
# >>> EDIT THIS LINE: Put your GitHub username and repo name <<<
GITHUB_REPO = "YOUR_GITHUB_USERNAME/artha"  # <-- CHANGE THIS

import os
if os.path.exists('/content/artha'):
    !rm -rf /content/artha

!git clone https://github.com/{GITHUB_REPO}.git /content/artha
os.chdir('/content/artha')
print(f"\n✅ Cloned to /content/artha")
!ls -la

## 2️⃣ Install Dependencies

In [ ]:
# =====================================================
# STEP 2: Install all Python dependencies
# =====================================================
# Colab already has numpy, pandas, scipy, scikit-learn.
# We install the trading-specific packages.

!pip install -q yfinance arch catboost lightgbm pypfopt vaderSentiment faiss-cpu jumpmodels

# Verify key imports
import yfinance, arch, catboost, lightgbm
print(f"\n✅ All dependencies installed")
print(f"   yfinance={yfinance.__version__}")
print(f"   arch={arch.__version__}")
print(f"   catboost={catboost.__version__}")
print(f"   lightgbm={lightgbm.__version__}")

## 3️⃣ Verify All 55 Modules Import Successfully

In [ ]:
# =====================================================
# STEP 3: Full import test
# =====================================================
import sys
sys.path.insert(0, '/content/artha')

modules = [
    # Layer 1 - Core
    'core.logger', 'core.time_utils',
    'config.settings', 'config.risk_settings',
    'engine.event_types', 'engine.message_bus', 'engine.clock',
    'engine.cache', 'engine.kernel',
    'database.connection',
    # Layer 2 - Data
    'adapters.data_feeds.yfinance_adapter',
    'input.market_data.ohlcv',
    'data_engine.ingest', 'data_engine.clean', 'data_engine.compute',
    'database.market_data_store',
    # Layer 3 - Alpha
    'alpha.base_alpha',
    'alpha.ml_model.catboost_model', 'alpha.ml_model.lightgbm_model',
    'alpha.ml_model.vader_gate',
    'alpha.volatility.egarch_model', 'alpha.volatility.gjrgarch_model',
    'alpha.volatility.vol_selector',
    'alpha.mean_reversion.ou_process', 'alpha.mean_reversion.kalman_filter',
    'strategy.regime.sjm', 'strategy.regime.bocpd',
    # Layer 4-6
    'risk.exposure', 'risk.pre_trade', 'risk.drawdown', 'risk.circuit_breaker',
    'portfolio.constructor', 'portfolio.optimizer', 'portfolio.rebalancer',
    'decision.signal_score', 'decision.conviction_score',
    'decision.ranking', 'decision.filtering', 'decision.prioritization',
    # Layer 7-9
    'backtest.engine', 'backtest.venue_sim',
    'backtest.validation.cpcv', 'backtest.validation.dsr_psr',
    'backtest.validation.monte_carlo',
    'execution.oms', 'execution.reconciler',
    'execution.execution_algos', 'execution.monitoring',
    'adapters.brokers.openalgo_adapter',
    'monitoring.ic_decay', 'monitoring.correlation',
]

import importlib, warnings
warnings.filterwarnings('ignore')

ok, fail = 0, 0
for m in modules:
    try:
        importlib.import_module(m)
        ok += 1
    except Exception as e:
        print(f"  ❌ {m}: {e}")
        fail += 1

print(f"\n{'✅' if fail == 0 else '❌'} {ok}/{ok+fail} modules imported successfully")

## 4️⃣ Run Full Backtest

In [ ]:
# =====================================================
# STEP 4: Run the full backtest via main.py
# =====================================================
!python main.py --mode backtest --tickers RELIANCE.NS TCS.NS INFY.NS HDFCBANK.NS ICICIBANK.NS --lookback 365

## 5️⃣ Run Paper Trading Session

In [ ]:
# =====================================================
# STEP 5: Run paper trading (latest signals)
# =====================================================
!python main.py --mode paper --tickers RELIANCE.NS TCS.NS INFY.NS

## 6️⃣ Test Individual Strategies

In [ ]:
# =====================================================
# STEP 6: Test individual strategy models
# =====================================================
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
from adapters.data_feeds.yfinance_adapter import fetch_ohlcv
from data_engine.ingest import ingest_raw_data

# Fetch data
raw = fetch_ohlcv(['RELIANCE.NS'], '2025-01-01', '2026-08-01')
bars = ingest_raw_data(raw)
records = [{'date': b.timestamp, 'close': b.close_price} for b in bars]
df = pd.DataFrame(records).set_index('date').sort_index()
returns = df['close'].pct_change().dropna() * 100  # scaled for arch

print(f"Data: {len(returns)} daily returns for RELIANCE.NS")
print(f"Mean: {returns.mean():.4f}%, Std: {returns.std():.4f}%")
print()

# --- Strategy 3: EGARCH ---
from alpha.volatility.egarch_model import EGARCHModel
egarch = EGARCHModel()
egarch.fit(returns)
vol = egarch.predict(returns)
print(f"Strategy 3 (EGARCH):  AIC={egarch.aic:.1f}, mean_vol={vol.mean():.4f}")

# --- Strategy 4: GJR-GARCH ---
from alpha.volatility.gjrgarch_model import GJRGARCHModel
gjr = GJRGARCHModel()
gjr.fit(returns)
vol_gjr = gjr.predict(returns)
print(f"Strategy 4 (GJR-GARCH): AIC={gjr.aic:.1f}, mean_vol={vol_gjr.mean():.4f}")

# --- Strategy 5: OU Process ---
from alpha.mean_reversion.ou_process import OUProcess
ou = OUProcess()
ou.fit(returns / 100)  # unscaled returns
ou_signals = ou.predict(returns / 100)
print(f"Strategy 5 (OU):      theta={ou.theta:.4f}, mu={ou.mu:.4f}")

# --- Strategy 7: SJM Regime ---
from strategy.regime.sjm import SJMRegimeDetector
features = pd.DataFrame({'ret': returns / 100})
features['vol'] = features['ret'].rolling(20).std()
features = features.dropna()
sjm = SJMRegimeDetector(n_components=2)
sjm.fit(features)
regimes = sjm.predict(features)
regime_counts = pd.Series(regimes).value_counts().to_dict()
print(f"Strategy 7 (SJM):     regimes={regime_counts}")

# --- Strategy 8: BOCPD ---
from strategy.regime.bocpd import BOCPD
bocpd = BOCPD()
bocpd.fit(returns / 100)
cp = bocpd.predict(returns / 100)
n_changes = sum(1 for i in range(1, len(cp)) if cp[i] != cp[i-1])
print(f"Strategy 8 (BOCPD):   changepoints={n_changes}")

print("\n✅ All strategy models run successfully on Colab!")

## 7️⃣ Risk & Validation

In [ ]:
# =====================================================
# STEP 7: Risk metrics & validation suite
# =====================================================
from risk.exposure import historical_cvar, historical_var
from backtest.validation.dsr_psr import probabilistic_sharpe_ratio
from backtest.validation.cpcv import combinatorial_purged_cv
from backtest.validation.monte_carlo import monte_carlo_permutation_test

daily_rets = (returns / 100).values  # unscaled

var95 = historical_var(pd.Series(daily_rets))
cvar95 = historical_cvar(pd.Series(daily_rets))
psr = probabilistic_sharpe_ratio(daily_rets)
mc = monte_carlo_permutation_test(daily_rets, n_simulations=500)
splits = combinatorial_purged_cv(n_samples=len(daily_rets), n_groups=5, n_test_groups=2, purge_gap=5)

print("=== Risk Metrics ===")
print(f"  VaR (95%):  {var95:.4%}")
print(f"  CVaR (95%): {cvar95:.4%}")
print(f"\n=== Validation ===")
print(f"  PSR:           {psr:.4f}")
print(f"  MC p-value:    {mc['p_value']:.4f}")
print(f"  CPCV splits:   {len(splits)}")
print("\n✅ Full validation suite works on Colab!")

## 8️⃣ Strategy Registry

In [ ]:
# =====================================================
# STEP 8: Verify strategy naming convention
# =====================================================
from alpha.ml_model.catboost_model import CatBoostAlpha
from alpha.ml_model.lightgbm_model import LightGBMAlpha
from alpha.volatility.egarch_model import EGARCHModel
from alpha.volatility.gjrgarch_model import GJRGARCHModel
from alpha.mean_reversion.ou_process import OUProcess
from alpha.mean_reversion.kalman_filter import KalmanFilterPairs
from strategy.regime.sjm import SJMRegimeDetector
from strategy.regime.bocpd import BOCPD

registry = [
    ('Strategy 1', 'CatBoost ML',    CatBoostAlpha().name),
    ('Strategy 2', 'LightGBM ML',    LightGBMAlpha().name),
    ('Strategy 3', 'EGARCH Vol',     EGARCHModel().name),
    ('Strategy 4', 'GJR-GARCH Vol',  GJRGARCHModel().name),
    ('Strategy 5', 'OU MeanRev',     OUProcess().name),
    ('Strategy 6', 'Kalman Pairs',   KalmanFilterPairs().name),
    ('Strategy 7', 'SJM Regime',     SJMRegimeDetector().name),
    ('Strategy 8', 'BOCPD Change',   BOCPD().name),
]

print(f"{'Expected':<15} {'Technique':<18} {'Code Name':<15} {'Match'}")
print('-' * 60)
all_ok = True
for expected, technique, actual in registry:
    code_name = expected.lower().replace(' ', '_')
    match = '✅' if actual == code_name else '❌'
    if actual != code_name: all_ok = False
    print(f"{expected:<15} {technique:<18} {actual:<15} {match}")

print(f"\n{'✅ All strategies correctly named!' if all_ok else '❌ Some names mismatch'}")

---

## ✅ Done!

The full ARTHA V1 system is running on Colab. All 8 strategies, the backtest engine, validation suite, risk framework, and OMS are functional.

**NO LIVE TRADING. Paper mode only. Founder decides go-live.**